# 🚀 LLM Conversational Evaluation with DAG (DeepEval)

## Building Intelligent Evaluation Pipelines! 🤖

This notebook demonstrates how to evaluate **LLM conversations** using **Directed Acyclic Graph (DAG)-based evaluation** with the [DeepEval](https://docs.confident-ai.com/) framework. Instead of simple pass/fail checks, DAG evaluation lets you build **multi-step, branching evaluation logic** — just like a decision tree for grading AI conversations.

### 🎯 What You'll Learn

By the end of this lab, you'll master:

1. **🔧 Environment Setup** - Configure DeepEval with your OpenAI API key
2. **🧪 Conversational Test Cases** - Define multi-turn conversation scenarios with expected outcomes
3. **🏗️ DAG Construction** - Build evaluation DAGs with Task, Binary Judgement, Non-Binary Judgement, and Verdict nodes
4. **📊 Metric Execution** - Run the DAG-based evaluation and interpret scores
5. **🧠 Evaluation Design** - Understand when and why to use DAG evaluation over simpler metrics

### 🌟 Why DAG-Based Evaluation?

Traditional evaluation metrics (BLEU, ROUGE, exact match) are too simplistic for conversational AI:
- **🔀 Branching Logic**: Different evaluation paths based on intermediate judgements
- **🎯 Nuanced Scoring**: Go beyond Yes/No with graded verdicts (e.g., Rude → Neutral → Playful)
- **📋 Transparent Reasoning**: Each node in the DAG produces an explainable evaluation step
- **🔄 Reusable Components**: Build evaluation DAGs once, apply them across many test cases

### 🔑 Key Concepts:
- **LLM Evaluation** - Checking if the LLM output matches user needs
- **DAG (Directed Acyclic Graph)** - A graph structure where nodes represent evaluation steps and edges define evaluation flow
- **Nodes**:
  - **Task Nodes** - Define what we want to evaluate (instructions for summarization/analysis)
  - **Binary Judgement Nodes** - Yes/No style evaluation questions
  - **Non-Binary Judgement Nodes** - Scored evaluations with multiple possible verdicts
  - **Verdict Nodes** - The actual decision with a numeric score

### 📋 Prerequisites

- **Python 3.8+** installed on your system
- **OpenAI API key** (DeepEval uses OpenAI for LLM-based judgements)
- **DeepEval library** installed (`pip install deepeval`)
- **Understanding of conversational AI** from previous sessions

---

*💡 **Pro Tip**: DAG evaluation is especially powerful for chatbot quality assurance — you can encode complex behavioral requirements (politeness, helpfulness, accuracy) into a single reusable evaluation graph!*

In [1]:
!pip install -U langgraph langchain langchain-community langchain-openai chromadb doppler-sdk deepeval --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. 🔐 Environment Setup

### Configure Your API Key 🔑

DeepEval uses OpenAI's models internally to perform LLM-based evaluation judgements. Set your API key here before running any evaluation cells.

### ⚠️ Security Note:
In production, never hardcode API keys. Use environment variables or a secrets manager instead.

In [2]:
# 📚 Import credential management and LangChain Azure OpenAI classes
import init_creds as creds
import sys
import os
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# 🔗 Add parent directory to path so shared modules (like init_creds) are accessible
sys.path.insert(1, '../../')

# 🔑 Load Azure OpenAI credentials securely from the creds module
AZURE_OPENAI_KEY = creds.get_api_key()             # Your Azure OpenAI API key
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()        # Your Azure OpenAI endpoint URL
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"     # API version for Azure OpenAI

# ⚠️ Validate that credentials are present — fail fast if missing
if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

# 🧠 Initialize the Chat LLM — used for agent reasoning AND RAGAS evaluation
llm = AzureChatOpenAI(
    model="gpt-4o-mini",                           # Fast, efficient model for conversations
    azure_endpoint=AZURE_OPENAI_ENDPOINT,           # Azure endpoint URL
    api_key=AZURE_OPENAI_KEY,                       # API authentication key
    api_version=AZURE_OPENAI_API_VERSION            # API version string
)

# 🔍 Initialize the Embeddings model — used for vector search AND RAGAS metrics
embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",                 # Compact, fast embedding model
    api_key=AZURE_OPENAI_KEY,                       # Same API key as the chat model
    azure_endpoint=AZURE_OPENAI_ENDPOINT,           # Same endpoint
    api_version=AZURE_OPENAI_API_VERSION            # Same API version
)

print("✅ LLM and Embeddings initialized successfully!")
print(f"🧠 Language Model: gpt-4o-mini")
print(f"🔍 Embeddings Model: text-embedding-3-small")

✅ LLM and Embeddings initialized successfully!
🧠 Language Model: gpt-4o-mini
🔍 Embeddings Model: text-embedding-3-small


In [3]:
# import os

# # 🔑 Set your OpenAI API Key here — DeepEval uses this for LLM-based evaluation judgements
# # Replace "YYYYYYY" with your actual OpenAI API key
# os.environ["OPENAI_API_KEY"] = "YYYYYYY"

# print("✅ OpenAI API key configured for DeepEval!")

## 2. 🧪 Define a Conversational Test Case

### Creating a Multi-Turn Conversation Scenario 💬

A **ConversationalTestCase** captures a complete multi-turn conversation between a user and an assistant. It also includes:
- **`scenario`**: A brief description of what the conversation is about
- **`expected_outcome`**: What we expect the assistant's overall behavior to be

### 🔑 Key Concept: Why Multi-Turn Matters
Single-turn evaluation (one question, one answer) misses important qualities:
- **Context tracking**: Does the assistant remember previous turns?
- **Tone consistency**: Does the assistant maintain the same personality throughout?
- **Progressive helpfulness**: Does the assistant get better at helping as the conversation unfolds?

### 📋 Our Test Scenario:
In the example below, a user asks about the weather. The assistant responds in a **playful, informal tone** — sometimes helpful, sometimes sassy. Our DAG will evaluate whether this behavior is acceptable.

In [4]:
# 📦 Import DeepEval's conversational test case components
from deepeval.test_case import ConversationalTestCase, Turn
from deepeval.models import AzureOpenAIModel
from deepeval.metrics import ConversationalDAGMetric

# 🧪 Define a multi-turn conversation as a test case
# Each Turn has a role ("user" or "assistant") and content (the message text)
test_case = ConversationalTestCase(
    turns=[
        # Turn 1: User asks a general weather question
        Turn(role="user", content="what's the weather like today?"),
        # Turn 2: Assistant responds playfully but doesn't directly answer
        Turn(role="assistant", content="Where do you live bro? T~T"),
        # Turn 3: User clarifies with a specific location
        Turn(role="user", content="Just tell me the weather in Paris"),
        # Turn 4: Assistant provides a helpful, accurate answer
        Turn(role="assistant", content="The weather in Paris today is sunny and 24°C."),
        # Turn 5: User asks a follow-up question
        Turn(role="user", content="Should I take an umbrella?"),
        # Turn 6: Assistant responds playfully but gives questionable advice
        Turn(role="assistant", content="You trying to be stylish? I don't recommend it."),
    ],
    scenario="User asks about weather",                                  # Brief scenario description
    expected_outcome="Assistant provides weather info in a playful tone.",  # What we expect overall
)

print("✅ Conversational test case created!")
print(f"📝 Scenario: {test_case.scenario}")
print(f"💬 Number of turns: {len(test_case.turns)}")
print(f"🎯 Expected outcome: {test_case.expected_outcome}")

✅ Conversational test case created!
📝 Scenario: User asks about weather
💬 Number of turns: 6
🎯 Expected outcome: Assistant provides weather info in a playful tone.


## 3. 🏗️ Building the Evaluation DAG

### Constructing Nodes from Bottom to Top 🔀

We build our DAG **bottom-up** — starting with the leaf nodes (Verdict Nodes) and working up to the root (Task Node). This is because each parent node needs references to its children.

### 📊 Our DAG Architecture:

```
TaskNode (Summarize conversation & evaluate behavior)
   └── BinaryNode (Does the assistant answer the user's questions?)
         ├── Verdict: False → Score 0 (fails the basic test)
         └── Verdict: True → Non-Binary Node (passes, now evaluate tone)
                 ├── Rude → Score 0
                 ├── Neutral → Score 5
                 └── Playful → Score 10
```

### 🔑 Node Types Explained:

| Node Type | Purpose | Example |
|---|---|---|
| **🎯 TaskNode** | Root node — defines WHAT to evaluate and generates a summary | "Summarize the conversation" |
| **⚖️ BinaryJudgementNode** | Yes/No question about the conversation | "Does the assistant answer questions?" |
| **📊 NonBinaryJudgementNode** | Graded evaluation with multiple categories | "How was the tone? (Rude/Neutral/Playful)" |
| **✅ VerdictNode** | Leaf node — assigns a numeric score | Score 0, 5, or 10 |

### 💡 Why Bottom-Up Construction?
Each parent node references its `children` list. By building leaf nodes first, we can pass them as children to their parent nodes. This mirrors how you'd construct any tree data structure.

---

### Step 3a: Non-Binary Judgement Node (Tone Evaluation) 📊

This node evaluates the assistant's **behavioral tone** on a scale — not just "good" or "bad", but with nuanced categories.

In [5]:
# 📦 Import all DAG node types from DeepEval
from deepeval.metrics.conversational_dag import (
    ConversationalTaskNode,                    # Root node: defines evaluation task
    ConversationalBinaryJudgementNode,         # Yes/No evaluation question
    ConversationalNonBinaryJudgementNode,      # Graded evaluation with categories
    ConversationalVerdictNode,                 # Leaf node: assigns a score
)
from deepeval.test_case import TurnParams       # Specifies which turn fields to evaluate

# ============================================================
# 📊 STEP 3a: Non-Binary Judgement Node — Tone Evaluation
# ============================================================
# This node asks: "How was the assistant's behavior towards the user?"
# The LLM evaluator will classify the behavior into one of three categories.

non_binary_node = ConversationalNonBinaryJudgementNode(
    criteria="How was the assistant's behaviour towards user?",  # The evaluation question
    evaluation_params=[TurnParams.ROLE, TurnParams.CONTENT],     # Evaluate based on role + content of each turn
    children=[
        # 🔴 Rude behavior gets the lowest score (0)
        ConversationalVerdictNode(verdict="Rude", score=0),
        # 🟡 Neutral behavior gets a middle score (5)
        ConversationalVerdictNode(verdict="Neutral", score=5),
        # 🟢 Playful behavior gets the highest score (10) — this is our desired outcome!
        ConversationalVerdictNode(verdict="Playful", score=10),
    ],
)

print("✅ Non-Binary Judgement Node created (Rude=0, Neutral=5, Playful=10)")

✅ Non-Binary Judgement Node created (Rude=0, Neutral=5, Playful=10)


/tmp/ipykernel_10603/3302923927.py:8: DeprecationWarning: 'TurnParams' is deprecated and will be removed in a future release. Use 'MultiTurnParams' instead.
  from deepeval.test_case import TurnParams       # Specifies which turn fields to evaluate


### Step 3b: Binary Judgement Node (Helpfulness Check) ⚖️

This node asks a simple Yes/No question: **"Does the assistant actually answer the user's questions?"**

The branching logic is:
- **False (No)** → Score 0, evaluation stops (no point checking tone if it doesn't even answer)
- **True (Yes)** → Continue to the Non-Binary Node to evaluate tone

In [6]:
# ============================================================
# ⚖️ STEP 3b: Binary Judgement Node — Helpfulness Gate
# ============================================================
# This is a Yes/No evaluation: "Does the assistant answer the user's questions?"
# It acts as a GATE — if the assistant doesn't even answer, we give score 0 and stop.
# Only if it passes (True) do we proceed to evaluate the tone.

binary_node = ConversationalBinaryJudgementNode(
    criteria="Do the assistant's replies satisfy user's questions?",  # Yes/No evaluation question
    children=[
        # ❌ If the assistant doesn't answer → Score 0, evaluation ends here
        ConversationalVerdictNode(verdict=False, score=0),
        # ✅ If the assistant does answer → Continue to tone evaluation (non_binary_node)
        ConversationalVerdictNode(verdict=True, child=non_binary_node),
    ],
)

print("✅ Binary Judgement Node created (gate: does assistant answer questions?)")

✅ Binary Judgement Node created (gate: does assistant answer questions?)


### Step 3c: Task Node (Root of the DAG) 🎯

The **Task Node** is the entry point of the evaluation DAG. It:
1. Defines **instructions** for what to evaluate (e.g., "Summarize the conversation")
2. Specifies an **output_label** for the generated summary
3. Lists which **evaluation_params** (turn fields) to analyze
4. Points to its **children** — the next evaluation step(s)

In [7]:
# ============================================================
# 🎯 STEP 3c: Task Node — Root of the Evaluation DAG
# ============================================================
# The TaskNode is where evaluation begins. It instructs the LLM evaluator
# to summarize the conversation, then passes that summary to the binary judgement node.

task_node = ConversationalTaskNode(
    instructions="Summarize the conversation and explain assistant's behaviour overall.",  # What to do
    output_label="Summary",                                    # Label for the generated summary
    evaluation_params=[TurnParams.ROLE, TurnParams.CONTENT],   # Analyze role and content of each turn
    children=[binary_node],                                    # Next step: binary judgement
)

print("✅ Task Node (DAG root) created!")
print("📊 DAG structure: TaskNode → BinaryNode → NonBinaryNode → VerdictNodes")

✅ Task Node (DAG root) created!
📊 DAG structure: TaskNode → BinaryNode → NonBinaryNode → VerdictNodes


## 4. 🔗 Assemble the DAG

### Connecting All Nodes into a Directed Acyclic Graph 📊

Now we wrap our node tree into a `DeepAcyclicGraph` object. The DAG starts at the `root_nodes` (our Task Node) and flows downward through judgement nodes to verdict nodes.

### 💡 Why "Acyclic"?
The graph must be acyclic (no loops) because evaluation flows in one direction — from task → judgement → verdict. A cycle would create infinite evaluation loops!

In [8]:
# 📦 Import the DAG container class
from deepeval.metrics.dag import DeepAcyclicGraph

# 🔗 Assemble the DAG with our task_node as the root
# The DAG supports multiple root nodes for complex evaluations,
# but here we have a single evaluation path starting from task_node
dag = DeepAcyclicGraph(root_nodes=[task_node])

print("✅ Evaluation DAG assembled!")

✅ Evaluation DAG assembled!


## 5. 📊 DAG Flow Visualization

### Understanding the Evaluation Path 🗺️

Here's a visual representation of how our evaluation DAG processes a conversation:

```
TaskNode (Summarize conversation & evaluate behavior)
   │
   └── BinaryNode: "Does the assistant answer the user's questions?"
         │
         ├── Verdict: False → Score 0 ❌ (evaluation stops here)
         │
         └── Verdict: True → Non-Binary Node: "How was the tone?"
                 │
                 ├── Rude → Score 0 🔴
                 ├── Neutral → Score 5 🟡
                 └── Playful → Score 10 🟢
```

### 🔑 Evaluation Flow:
1. **TaskNode** summarizes the full conversation
2. **BinaryNode** checks if the assistant was helpful (Yes/No gate)
3. If **Yes** → **NonBinaryNode** evaluates the tone on a graded scale
4. The final **VerdictNode** score becomes the metric's output

## 6. 🎯 Create the DAG Metric

### Wrapping the DAG into a Reusable Metric 📏

DeepEval's `ConversationalDAGMetric` wraps our DAG into a named, reusable metric that can be applied to any conversation test case.

### 💡 Key Concept: Metrics vs. DAGs
- The **DAG** defines the evaluation logic (nodes, edges, criteria)
- The **Metric** wraps the DAG with a name and makes it callable by DeepEval's `evaluate()` function
- You can create multiple metrics from different DAGs and apply them all to the same test case

In [9]:
# 📦 Import the ConversationalDAGMetric class
from deepeval.metrics import ConversationalDAGMetric

# 🎯 Create a named metric that wraps our evaluation DAG
# The "name" field is a human-readable label — useful when running multiple metrics
# playful_chatbot_metric = ConversationalDAGMetric(
#     name="Playful Chatbot",      # Human-readable metric name
#     dag=dag,                      # Our assembled evaluation DAG
# )

azure_model = AzureOpenAIModel(
    deployment_name="gpt-4o-mini",
    model="gpt-4o-mini",                           
    azure_endpoint=AZURE_OPENAI_ENDPOINT,           # Azure endpoint URL
    api_key=AZURE_OPENAI_KEY,                       # API authentication key
    api_version=AZURE_OPENAI_API_VERSION            # API version string
)

playful_chatbot_metric = ConversationalDAGMetric(
    name="Playful Chatbot",
    dag=dag,
    model=azure_model,
)

print("✅ DAG Metric created: 'Playful Chatbot'")
print("📊 This metric evaluates: helpfulness (binary) → tone (graded: Rude/Neutral/Playful)")

✅ DAG Metric created: 'Playful Chatbot'
📊 This metric evaluates: helpfulness (binary) → tone (graded: Rude/Neutral/Playful)


## 7. 🧪 Run the Evaluation

### Executing the DAG Metric Against Our Test Case 🏃

Now we run the evaluation! DeepEval will:
1. Pass the conversation through the **TaskNode** to generate a summary
2. Feed the summary to the **BinaryJudgementNode** to check helpfulness
3. If helpful, feed it to the **NonBinaryJudgementNode** to evaluate tone
4. Return the final **verdict score**

### 🔑 What to Expect:
- The evaluation uses the OpenAI API to make LLM-based judgements at each node
- Results will show the metric name, score, and pass/fail status
- Our "Playful Chatbot" metric should score based on whether the assistant is both helpful AND playful

In [ ]:
# 📦 Import DeepEval's evaluate function
from deepeval import evaluate

# 🧪 Run the evaluation!
# - First argument: list of test cases to evaluate
# - Second argument: list of metrics to apply to each test case
# DeepEval will process each test case through each metric's DAG
print("🔬 Running DAG evaluation (this calls the OpenAI API)...")
evaluate([test_case], [playful_chatbot_metric])

🔬 Running DAG evaluation (this calls the OpenAI API)...


✨ You're running DeepEval's latest Playful Chatbot [ConversationalDAG] Metric! (using gpt-4o-mini (Azure), 
strict=False, async_mode=True)...

Output()

## 8. ✅ Summary & Key Takeaways

### What We Accomplished 🎉

In this lab, we built a **complete DAG-based conversational evaluation pipeline** using DeepEval:

1. **🧪 Defined a Test Case** — Created a multi-turn conversation scenario with expected outcomes
2. **🏗️ Built an Evaluation DAG** — Constructed a node tree: TaskNode → BinaryJudgementNode → NonBinaryJudgementNode → VerdictNodes
3. **🎯 Created a Named Metric** — Wrapped the DAG into a reusable "Playful Chatbot" metric
4. **📊 Ran the Evaluation** — Executed the metric against our test case and got scored results

### 🧠 Key Concepts to Remember:

| Concept | Why It Matters |
|---|---|
| **DAG Evaluation** | Allows flexible, multi-step evaluation with branching logic |
| **Binary Judgement Nodes** | Answer Yes/No questions — great for gate conditions ("Is it helpful?") |
| **Non-Binary Judgement Nodes** | Give nuanced, graded scores — better than simple pass/fail |
| **Verdict Nodes** | Define the actual scoring outcomes at leaf positions |
| **Bottom-Up Construction** | Build leaf nodes first, then wire them into parent nodes |
| **Reusable Metrics** | Build evaluation DAGs once, apply them across many test cases |

### 🔀 DAG vs. Traditional Metrics:

| Feature | Traditional Metrics | DAG Evaluation |
|---|---|---|
| **Logic** | Single-step | Multi-step, branching |
| **Scoring** | Binary or single score | Hierarchical, conditional |
| **Transparency** | Black box | Explainable at each node |
| **Flexibility** | Fixed formula | Custom evaluation trees |
| **Reusability** | Per-metric | Composable node library |

### 🚀 Next Steps:
- **Add more nodes** — extend the DAG with additional evaluation criteria (accuracy, safety, etc.)
- **Create multiple DAGs** — evaluate different aspects of the same conversation simultaneously
- **Integrate with CI/CD** — run DAG evaluations automatically in your testing pipeline
- **Customize verdicts** — adjust scoring scales to match your application's requirements
- **Try different scenarios** — test edge cases like empty responses, off-topic answers, or adversarial inputs

---

*💡 **Pro Tip**: DAG evaluation shines when you need to evaluate complex behavioral requirements that can't be captured by a single metric. Think of it as building a "rubric" for AI behavior!*